# Reichman University NLP Project

Cross-request **token reuse** across LLM workloads. This notebook runs one
experiment per corpus and assembles the results table.

Each experiment lives in `experiments/`; the shared measurement core is in
`core/`. A corpus whose data isn't available locally falls back to the
published value, so the table always renders. HuggingFace-hosted corpora
download on first run — set `HF_TOKEN` and expect the coding/chat rows to be
the slow ones.


In [ ]:
import contextlib, importlib.util, io, os, sys
import pandas as pd

ROOT = os.getcwd()                       # run this notebook from the repo root
EXP = os.path.join(ROOT, "experiments")
sys.path.insert(0, EXP)                  # so each experiment's `import _bootstrap` resolves
import _bootstrap                        # puts core/ on the path and pins the working dir
print("repo:", ROOT)

In [ ]:
# family, corpus, module, treatment, published = (prefix, PIC, PIC-proc, in-sess, cross-sess)
TABLE = [
    ("Chat prompts",        "WildChat-1M (40K)",              "wildchat",        "none",     ("26.62", 3.28,  3.28,  0.02,  3.27)),
    ("Paraphrase traffic",  "PAWS (40K)",                     "paws",            "none",     ("27.67", 0.00,  0.00,  0.00,  0.00)),
    ("Coding agents",       "SWE-smith (1K)",                 "swe_smith",       "none",     ("4.21",  17.06, 17.06, 2.43,  14.64)),
    ("Coding agents",       "OpenHands (1K)",                 "openhands",       "none",     ("6.42",  2.69,  2.69,  1.87,  0.82)),
    ("Coding agents",       "CC-Bench (74)",                  "ccbench",         "none",     ("0.00",  16.43, 16.43, 7.79,  8.64)),
    ("Coding agents",       "SWE-agent (1K)",                 "swe_agent",       "none",     ("7.96",  19.27, 19.27, 18.99, 0.28)),
    ("Web pages (a11y)",    "NNetNav-WA (WebArena)",          "nnetnav_wa",      "replace",  ("37.32", 4.50,  10.03, 5.62,  4.41)),
    ("Web pages (HTML)",    "Mind2Web",                       "mind2web",        "replace",  ("0.71",  0.00,  42.94, 34.80, 8.13)),
    ("Web pages (live)",    "NNetNav-Live",                   "nnetnav_live",    "replace",  ("20.31", 11.82, 14.79, 11.79, 3.00)),
    ("Op. telemetry",       "ITBench SRE",                    "itbench_sre",     "relocate", ("0.18",  0.00,  7.71,  7.71,  0.00)),
    ("Op. telemetry",       "ITBench infra compliance (K8s)", "itbench_k8s",     "none",     ("0.71",  26.73, 26.73, 26.73, 0.00)),
    ("Op. telemetry",       "ITBench alerts",                 "itbench_alerts",  "none",     ("0.08",  16.93, 16.93, 16.93, 0.00)),
    ("Op. telemetry",       "BGL syslog (LogHub)",            "bgl_syslog",      "relocate", ("0.00",  0.00,  8.49,  4.53,  3.96)),
    ("Stateful API",        "AppWorld",                       "appworld",        "mask",     ("16.93", 0.21,  0.40,  0.00,  0.40)),
    ("Stateful API",        "tau2 airline",                   "tau2_airline",    "none",     ("12.81", 1.31,  1.31,  1.31,  0.00)),
    ("Stateful API",        "tau2 retail",                    "tau2_retail",     "none",     ("23.28", 23.08, 23.08, 0.33,  22.75)),
    ("Stateful API",        "tau2 telecom",                   "tau2_telecom",    "none",     ("60.70", 3.73,  3.73,  0.00,  3.73)),
    ("Retrieved docs",      "MT-RAG ibmcloud",                "mtrag_ibmcloud",  "none",     ("0.44",  13.76, 13.76, 12.86, 0.90)),
    ("Retrieved docs",      "MultiDoc2Dial",                  "multidoc2dial",   "none",     ("56.01", 39.97, 39.97, 3.04,  36.92)),
    ("Retrieved docs",      "tau-Knowledge",                  "tau_knowledge",   "align",    ("34.77", 4.32,  47.74, 0.00,  47.74)),
    ("Database schemas",    "Spider 2.0",                     "spider2",         "none",     ("86.5/0.0", 86.95, 86.95, 0.00, 86.95)),
    ("Database schemas",    "BIRD",                           "bird",            "none",     ("93.7/0.0", 78.54, 78.54, 0.00, 78.54)),
    ("Database schemas",    "LiveSQLBench",                   "livesqlbench",    "none",     ("95.9/0.0", 95.86, 95.86, 0.00, 95.86)),
    ("Conversational SQL",  "BIRD-INTERACT",                  "bird_interact",   "none",     ("97.9/0.0", 97.87, 97.87, 1.84, 96.03)),
]

RUN = True                 # False -> just render the published table, no compute
ONLY = None                # e.g. {"bgl_syslog", "tau2_retail"} to run a subset

In [ ]:
def run_module(mod):
    path = os.path.join(EXP, mod + ".py")
    spec = importlib.util.spec_from_file_location("exp_" + mod, path)
    m = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(m)
    _bootstrap.RESULTS.clear()
    with contextlib.redirect_stdout(io.StringIO()):
        m.main()
    return next(iter(_bootstrap.RESULTS.values()))["computed"]

def overlay(pub, c):
    prefix, pic, proc, ins, crs = pub
    if "prefix_caching_pct" in c:
        prefix = c["prefix_caching_pct"]
    elif "prefix_schema_first" in c:
        prefix = f'{c["prefix_schema_first"]}/{c["prefix_schema_after"]}'
    return (prefix, c.get("pic_pct", pic), c.get("pic_proc_pct", c.get("pic_pct", proc)),
            c.get("in_session", ins), c.get("cross_session", crs))

rows = []
for fam, corpus, mod, treat, pub in TABLE:
    vals, src = pub, "published"
    if RUN and (ONLY is None or mod in ONLY):
        try:
            vals, src = overlay(pub, run_module(mod)), "recomputed"
        except Exception as e:
            src = f"published ({type(e).__name__})"
    print(("+ " if src == "recomputed" else "- ") + f"{corpus:32} {src}")
    rows.append([fam, corpus, *vals, treat, src])

cols = ["Workload family", "Corpus", "Prefix caching", "PIC", "PIC proc.",
        "in-sess.", "cross-sess.", "Treatment", "Source"]
df = pd.DataFrame(rows, columns=cols)

In [ ]:
# bold the best PIC-proc corpus within each workload family, as in the paper
def bold_family_best(col):
    best = df.groupby("Workload family")["PIC proc."].transform("max")
    return ["font-weight: bold" if v == b else "" for v, b in zip(col, best)]

df.style.apply(bold_family_best, subset=["PIC proc."]).format(precision=2).hide(axis="index")

**Columns.** *Prefix caching* is the share ordinary prefix caching already
serves; *PIC* is position-independent reuse beyond it; *PIC proc.* is PIC after
the corpus's *Treatment* (`replace` / `relocate` / `mask` / `align`, or `none`),
split into its in-session and cross-session parts. PIC savings are **in addition**
to prefix caching. `Source` marks whether the row was recomputed here or fell
back to the published value because its data wasn't available.
